# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.text_processing_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *
from src.utils import *



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# load data (model)

res_savename = "test_reports_gaps_chunksize800_2it_meta-llama_llama-4-scout-17b-16e-instruct_v090426"
# "labelled_reports_turnoff_subtype_val_llama-3.3-70b-versatile_v141025.csv"
# "labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"
response_df = pd.read_csv(DATA_OUT_LLMS /(res_savename + ".csv"))
# load data (labelled)
# res_savename = "labelled_reports_impacts_all_v080925.csv"
# response_df = pd.read_csv(DATA_LABELLED / res_savename)
savename = "nounit_quali_post_processed_" + res_savename

In [3]:
#get rid of nans
response_df_proc = cp.deepcopy(response_df)

response_df_proc = response_df_proc.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df_proc.columns else response_df_proc

In [6]:
#process impactValue
response_df_proc = response_df_proc.apply(parse_impact_value_precision, axis=1)

In [8]:
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation", "annotation", "nathaz_text"]
list_cols = [key for key in list_cols if key in response_df_proc.columns]
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [10]:
appeal_text = response_df_proc[response_df_proc["appealCode"] == "MDRYE011"]["nathaz_text"].values[0]

In [15]:
from src.LLM_functions import break_down_text
chunk_size = 800
chunks = break_down_text(appeal_text, chunk_size)

In [16]:
chunks

[["P a g e | 1 Internal DREF Operation n° MDRYE011 Glide n°: FL-2022-000265-YEM Date of issue: 29/07/2022 Expected timeframe: 6 months Expected end date: 31/01/2023 Category allocated to the of the disaster or crisis: Orange DREF allocated: CHF 452,156 Total number of people affected: Approximately 76,790 people affected Number of people to be assisted: 19,509 people (2,787 HH) Governorates affected: Marib, Al Mahwit, Taiz, Ibb, Hadramawt, Al Bayda, Amran, Sadaa, Dhamar Al Hodeida Sana'a Hajjah, Al Mahra governorates Governorates targeted: Al Hodeida, Hajjah Hadramout, and Al- Mahra, Marib and Sana’a Governorates Operating National Society: Yemen Red Crescent Society has branches in all 22 Governates of Yemen, with 321 staff and 4,500 active volunteers, including 44 National Disaster Response"],
 [' trained team members, as well as trained first aid volunteers ready to deploy in case of emergency.',
  'Red Cross Red Crescent Movement partners: British Red Cross, Danish Red Cross, Germa

In [21]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(list_country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(list_country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [22]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [27]:
response_df.loc[pd.isnull(response_df["impactUnit"]),"impactUnit"] = "null"

In [33]:
import regex as re
response_df[response_df.apply(lambda x: "destroyed" in x["impactUnit"],axis=1)]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,country,location,...,valid_errors_dates,hazards,hazardsAnnotation,valid_errors_haz,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
423,Residential Buildings,6.0,NaN,NaN,exact,other buildings destroyed (collapsed walls),['Other buildings destroyed (collapsed walls) 6'],0,['El Salvador'],NaN,...,0,"['Flood', 'Tropical storm']","['On 6 October, rains began falling over easte...",0,MDRSV012,El Salvador,2019-06-26,https://adore.ifrc.org/Download.aspx?FileId=24...,Flood,['DREF Operation n° MDRSV012 GLIDE: n° TC-2018...
460,Other Infrastructure,6.0,NaN,NaN,exact,other buildings destroyed (collapsed walls),['Other buildings destroyed (collapsed walls) 6'],0,['El Salvador'],NaN,...,0,"['Flood', 'Tropical storm']","['On 6 October, rains began falling over easte...",0,MDRSV012,El Salvador,2019-06-26,https://adore.ifrc.org/Download.aspx?FileId=24...,Flood,['DREF Operation n° MDRSV012 GLIDE: n° TC-2018...


In [27]:
#reclassify impacType
#response_df_proc["impactSubtype_orig"] = response_df_proc["impactSubtype"]
response_df_proc = response_df_proc.apply(reclassify_impact_subtype, axis=1)
response_df_proc = response_df_proc[response_df_proc["impactSubtype"] != "Unknown"]


In [28]:
response_df_proc["impactSubtype"].value_counts()

impactSubtype
Affected People                                    44
Other Human Impacts                                40
Residential Buildings                              37
Other Infrastructural Impacts                      26
Crop Production and Forestry                       23
Human Health and Wellbeing                         22
Access to Food                                     20
Access to Water, Sanitation, and Hygiene           20
Displaced People                                   19
Access to Healthcare                               18
Homeless People                                    16
Affected Livestock and Animals                     15
Water Quality and Availability                     15
Human Deaths                                       14
Agricultural Infrastructure                        13
Other Economic Activity & Livelihood Production    13
Road Infrastructure                                10
Other Service Access Impacts                        9
Water, Sanitat

In [29]:
#reclassify hazard
response_df_proc["hazards_orig"] = response_df_proc["hazards"]
response_df_proc = response_df_proc.apply(reclassify_hazard, hazard_kw_reclass=hazard_kw_reclass, axis=1)
response_df_proc.hazards.value_counts()

hazards
[Flood]                                                                                                                122
[Flood, Convective storm]                                                                                               46
[Drought]                                                                                                               41
[Flood, Mass movement]                                                                                                  26
[Flood, Tropical storm]                                                                                                 25
[Flood, Epidemic, Conflict]                                                                                             20
[Flood, Epidemic]                                                                                                       18
[Drought, Tropical storm]                                                                                               17
[Tropica

In [30]:
response_df_proc[response_df_proc["appealCode"] == "MDRSD034"][["hazards","hazards_orig", "flag_hazards_reclass"]]

,hazards,hazards_orig,flag_hazards_reclass
27,"[Flood, Epidemic]","[Flood, Epidemic]",False
28,"[Flood, Epidemic]","[Flood, Epidemic]",False
29,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
30,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
31,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
32,"[Flood, Epidemic]","[Flood, Epidemic]",False
33,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
34,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
35,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
36,[Flood],[Flood],False


In [31]:
response_df_proc

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,country,location,...,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,flag_impactSubtype_reclass,hazards_orig,flag_hazards_reclass
0,Affected People,326788.0,NaN,NaN,exact,people,"[People Affected: 326,788 people]",0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
1,Injured People,584.0,NaN,NaN,exact,people,[The monsoon season caused 306 fatalities and ...,0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh, Punja...",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,"[Flood, Convective storm]",False
2,Human Deaths,306.0,NaN,NaN,exact,people,[The monsoon season caused 306 fatalities and ...,0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh, Punja...",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,"[Flood, Convective storm]",False
3,Displaced People,9500.0,NaN,NaN,exact,residents,"[Sindh experienced acute urban flooding, parti...",0,[Pakistan],"[Badin, Dadu, Jacobabad]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
4,Homeless People,15000.0,NaN,NaN,exact,houses,"[In response to this situation, the government...",0,[Pakistan],"[Sindh, Balochistan, Khyber Pakhtunkhwa]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,Other Human Impacts,1.0,NaN,NaN,exact,NaN,[The original plan was to hire an appeal coord...,1,[Guatemala],"[El Quiché, Western Temperate Highlands]",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
484,Other Human Impacts,130.0,NaN,NaN,exact,training sessions,[From 130 to 50 training sessions in hygiene p...,0,[Guatemala],"[El Quiché, Western Temperate Highlands, Guate...",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
485,Other Human Impacts,13.0,NaN,NaN,exact,%,"[Appeal Coverage to date: 13 % (269,543 CHF)]",0,[Guatemala],"[El Quiché, Western Temperate Highlands]",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
486,Crop Production and Forestry,NaN,NaN,NaN,approx,NaN,[According to the Food Security Outlook Update...,2,[Guatemala],"[Western Temperate Highlands, low-lying areas ...",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False


In [32]:
wrong_haz = response_df_proc.explode("hazards", ignore_index=True).copy()
wrong_haz = wrong_haz[wrong_haz["hazards"] == "Unknown"]
wrong_haz["hazards"]

Series([], Name: hazards, dtype: object)

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent) and special units (money, deaths)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people
3. ~~Handle currencies~~
4. When unknown unit, try to infer it using kw for other impact subtype and reclass to other subtype if match
5. Handle damage vs destroyed houses
6. Parse correctly impact values min and maxs

In [33]:
#response_df_proc["impactValue"] = response_df_proc["impactValueOrig"]
#response_df_proc["impactUnit"] = response_df_proc["impactUnitOrig"]

In [34]:
def infer_unit_from_annotation(x):
    annotation = x["annotation"] if x["annotation"] else x["valueAnnotation"]
    value = x["impactValue"]
    unit = x["impactUnit"]
    #if there is no value we return what's orignally there
    if pd.isnull(value):
        return pd.Series({"impactValue": value, "impactUnit": unit})

    #format numbers in annotation
    annotation = replace_commas_in_numbers(annotation)
    annotation = replace_count_suffixes(annotation)
    annotation = replace_numbers(annotation)

    #format value
    value = format_number(value)

    #find value in annotation
    kw_value = re.search(r"\{value\}", annotation, re.IGNORECASE)
    unit_kw = [kw for kw in unit_kw_reclass.keys() if re.search(unit_kw_reclass[kw], annotation, re.IGNORECASE)]
    if len(unit_kw) == 1:
        return pd.Series({"impactValue": value, "impactUnit": unit_kw[0]})
    else:
        return "Unknown"


In [35]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
reclass_subtype = True
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
## Units reclassification
#replace numbers in units
response_df_proc = response_df_proc.apply(replace_numbers_unit, axis=1)
#convert money
response_df_proc = response_df_proc.apply(convert_monetary_units, axis=1)
#standardize metric units
response_df_proc = response_df_proc.apply(standardize_metric_units, axis=1)
#assign unit type (e.g. surface, volume, mass)
response_df_proc = response_df_proc.apply(assign_unit_type, axis=1)
#harmonize non metric units
response_df_proc = response_df_proc.apply(harmonize_units, axis=1)
#convert convertible (non-money) units
response_df_proc = response_df_proc.apply(convert_unit, axis=1)
#reclassify units
response_df_proc = response_df_proc.apply(reclassify_units, force_unit_to_subtype=force_unit_to_subtype, reclass_subtype=reclass_subtype, axis=1)
#normalize people units
response_df_proc = response_df_proc.apply(normalize_people_unit, axis=1)


/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:536: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  if len(pd.unique(units_parsed)) > 1:#only do assignment if all units are the same
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:539: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  elif len(pd.unique(units_parsed)) == 1:
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:536: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  if len(pd.unique(units_parsed)) > 1:#only do assignment if all units are the same
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/sr

Reclassified subtype from Homeless People to Residential Buildings with unit reclass homes and orig unit homes
Reclassified subtype from Agricultural Infrastructure to Affected Livestock and Animals with unit reclass affected animals and orig unit livestock
Reclassified subtype from Road Infrastructure to Displaced People with unit reclass displaced and orig unit people displaced
Reclassified subtype from Water Quality and Availability to Infected and Ill People with unit reclass cases and orig unit cases
Reclassified subtype from Other Infrastructural Impacts to Water, Sanitation, and Hygiene Infrastructure with unit reclass WASH structures and orig unit latrines
Reclassified subtype from Water Quality and Availability to Water, Sanitation, and Hygiene Infrastructure with unit reclass WASH structures and orig unit water treatment plants
Reclassified subtype from Other Infrastructural Impacts to Road Infrastructure with unit reclass roads and orig unit roads
Reclassified subtype from O

In [19]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
reclass_subtype = True
test_df = pd.DataFrame({"impactUnit": "nomads families", "unit_type": "%", "impactValue": 100, "impactValueMin": np.nan, "impactValueMax":200}, index=[0])
#harmonize non metric units
test_df = test_df.apply(harmonize_units, axis=1)
#convert convertible (non-money) units
test_df = test_df.apply(convert_unit, unit_converter=unit_converter, axis=1)
#reclassify units
test_df = test_df.apply(reclassify_units, unit_kw_reclass=unit_kw_reclass, default_subtype_unit=default_subtype_unit, force_unit_to_subtype=force_unit_to_subtype, reclass_subtype=reclass_subtype, axis=1)
#normalize people units
test_df = test_df.apply(normalize_people_unit, axis=1)
test_df

,impactUnit,unit_type,impactValue,impactValueMin,impactValueMax,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit
0,nomads people,%,300.0,NaN,600.0,False,True,True,False


In [20]:
test_df

,impactUnit,unit_type,impactValue,impactValueMin,impactValueMax,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit
0,nomads people,%,300.0,NaN,600.0,False,True,True,False


In [22]:
## Post conversion flags
country_pop = pd.read_csv(DATA_PATH / ("API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2).dropna(how="all",axis=1)
# country_pop = pd.read_csv(os.path.join(DATA_PATH, "API_SP.POP.TOTL_DS2_en_csv_v2_131993", "API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2).dropna(how="all",axis=1)
response_df_proc["flag_pop_cntry"] = response_df_proc.apply(pop_cntry_check, country_pop=country_pop, axis=1)
response_df_proc["flag_value_no_unit"] = response_df_proc.apply(flag_value_no_unit, axis=1)
response_df_proc["flag_partial_unit"] = response_df_proc.apply(flag_partial_unit, axis=1)
response_df_proc["flag_percent"] = response_df_proc.apply(flag_percent, axis=1)

 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023


In [23]:
# save
response_df_proc.to_csv(DATA_OUT_PROC / savename, index=False)

## Post post process

In [14]:
#load data (model)
res_savename = "post_processed_new_unit_std_labelled_reports_impacts_all_v111025"
#"post_processed_all_appeals_unique_1-333_meta-llama_llama-4-scout-17b-16e-instruct_v281025"
#"post_processed_labelled_reports_fixed_impact_desc_meta-llama_llama-4-scout-17b-16e-instruct_v271025"
#"post_processed_labelled_reports_DREF_target_meta-llama_llama-4-scout-17b-16e-instruct_v211025"
#"post_processed_new_unit_std_labelled_reports_impacts_all_v111025"
#"labelled_reports_turnoff_subtype_val_llama-3.3-70b-versatile_v141025.csv"
#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"
#response_df = pd.read_csv(DATA_OUT_LLMS /(res_savename + ".csv"))
suffix = "_geo_split_lowest_v171025"
res_savename_geo = res_savename + suffix
response_df = pd.read_csv(DATA_OUT_PROC / (res_savename + ".csv"))
response_df_geo = gpd.read_file(DATA_OUT_PROC / (res_savename_geo+".gpkg"))
#load data (labelled)
#res_savename = "labelled_reports_impacts_all_v080925.csv"
#response_df = pd.read_csv(DATA_LABELLED / res_savename)
savename = "merged_subtypes_"

In [15]:
if "quanti" not in response_df_geo.columns:
    response_df_geo = response_df_geo.apply(label_quanti_quali, axis=1)

response_df_geo.loc[response_df_geo["quanti"] == "quali", "impactUnit"] = "null"

In [16]:
if "quanti" not in response_df.columns:
    response_df = response_df.apply(label_quanti_quali, axis=1)

response_df.loc[response_df["quanti"] == "quali", "impactUnit"] = "null"

In [17]:
#merge infra and service access
response_df_geo = response_df_geo.apply(merge_impact_subtypes, impact_kw_reclass=IMPACT_SUBTYPE_MERGER,axis=1)
response_df = response_df.apply(merge_impact_subtypes, impact_kw_reclass=IMPACT_SUBTYPE_MERGER,axis=1)



In [18]:
response_df_geo[response_df_geo["impactSubtype"] == "Undefined Infrastructure"]

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,...,locationLowestAdmin,geocoding_country_flag,geocoding_osm_flag,locationOsm,locationPolygon,iso3_code,impactValueApprox,geometry,quanti,flag_impactSubtype_merged


In [19]:
response_df_geo.impactSubtype.value_counts()

impactSubtype
Residential Buildings                                                                         133
Affected People                                                                               123
Agriculture and Access to Food                                                                 85
Transportation Infrastructure and Access to Mobility                                           50
Water, Sanitation, and Hygiene Infrastructure and Access to Water, Sanitation, and Hygiene     48
Displaced People                                                                               47
Undefined Infrastructure and Service Access                                                    34
Infected and Ill People                                                                        32
Human Deaths                                                                                   29
Other Economic Activity & Livelihood Production                                                26
Health

In [20]:
from src.geocoding import atomic_gpkg_save
atomic_gpkg_save(response_df_geo, DATA_OUT_PROC / (savename + res_savename_geo + ".gpkg"))
response_df.to_csv(DATA_OUT_PROC / (savename + res_savename + ".csv"), index=False)